In [ ]:
import sqlite3
import pandas as pd
import os


df = pd.read_csv('../Data/Cleaned/cleaned_data.csv')

db_path = '../Data/Cleaned/bankruptcy.db'

conn = sqlite3.connect(db_path)
cur = conn.cursor()

In [ ]:
companies = df[['company_name']].drop_duplicates().reset_index(drop=True)

companies['company_id'] = companies.index+1

companies = companies[['company_id', 'company_name']]

companies.head()

,company_id,company_name
0,1,C_1
1,2,C_2
2,3,C_3
3,4,C_4
4,5,C_5


In [27]:
macro_econ = df[['year', 'cpi', 'unemployment_rate', 'baa_rate']].drop_duplicates().reset_index(drop=True)

df_new = df.merge(companies, on= 'company_name', how='left')

financials = df_new[[
    "company_id",
    "year",
    "status_label",
    "ebitda",
    "net_income",
    "market_value",
    "long_term_debt",
    "retained_earnings",
    "total_revenue",
    "current_ratio",
    "debt_ratio",
    "net_profit_margin",
    "asset_turnover_ratio",
    "inventory_turnover",
    "interest_rate_pressure"
]]

In [29]:
cur.execute("""
CREATE TABLE IF NOT EXISTS companies (
    company_id INTEGER PRIMARY KEY,
    company_name TEXT UNIQUE NOT NULL
);
""")


cur.execute("""
    CREATE TABLE IF NOT EXISTS macro_econ (
        year INTEGER PRIMARY KEY,
        cpi REAL,
        unemployment_rate REAL,
        baa_rate REAL
    );
    """)

cur.execute("""
    CREATE TABLE IF NOT EXISTS financials (
        company_id INTEGER,
        year INTEGER,
        status_label TEXT,
        ebitda REAL,
        net_income REAL,
        market_value REAL,
        long_term_debt REAL,
        retained_earnings REAL,
        total_revenue REAL,
        current_ratio REAL,
        debt_ratio REAL,
        net_profit_margin REAL,
        asset_turnover_ratio REAL,
        inventory_turnover REAL,
        interest_rate_pressure REAL,
        
        PRIMARY KEY (company_id, year),
        FOREIGN KEY (company_id) REFERENCES companies (company_id),
        FOREIGN KEY (year) REFERENCES macro_econ (year)
    );
    """)

conn.commit()

In [30]:
companies.to_sql('companies', conn, if_exists='replace', index=False)
macro_econ.to_sql('macro_econ', conn, if_exists='replace', index=False)
financials.to_sql('financials', conn, if_exists='replace', index=False)

78682

In [37]:
# test

query = """
SELECT
    c.company_name,
    f.year,
    f.status_label,
    f.market_value,
    f.net_income,
    f.total_revenue,
    m.cpi,
    m.unemployment_rate,
    m.baa_rate
FROM financials f
JOIN companies c
    ON f.company_id = c.company_id
JOIN macro_econ m
    ON f.year = m.year
LIMIT 10;
"""

test_join = pd.read_sql(query, conn)

test_join

,company_name,year,status_label,market_value,net_income,total_revenue,cpi,unemployment_rate,baa_rate
0,C_1,1999,alive,372.7519,35.163,1024.333,166.583333,4.216667,2.229124
1,C_1,2000,alive,377.1180,18.531,874.255,172.191667,3.966667,2.336693
2,C_1,2001,alive,364.5928,-58.939,638.721,177.041667,4.741667,2.925628
3,C_1,2002,alive,143.3295,-12.410,606.337,179.866667,5.783333,3.189760
4,C_1,2003,alive,308.9071,3.504,651.958,184.000000,5.991667,2.748680
5,C_1,2004,alive,522.6794,15.453,747.848,188.908333,5.541667,2.120640
6,C_1,2005,alive,882.6283,35.163,897.284,195.266667,5.083333,1.773200
7,C_1,2006,alive,1226.1925,58.660,1061.169,201.558333,4.608333,1.687960
8,C_1,2007,alive,747.5434,75.144,1384.919,207.344167,4.616667,1.847490
9,C_1,2008,alive,571.5948,78.651,1423.976,215.254250,5.800000,3.773068


In [38]:
conn.close()